# FastText ile Işık Hızında Metin Sınıflandırma

Bu notebook'ta Meta/Facebook tarafından geliştirilen **FastText** kütüphanesini kullanacağız.
FastText, derin öğrenme modelleri (BERT, LSTM) kadar yüksek performans gösterirken, saniyeler içinde eğitilebilen ve CPU üzerinde bile inanılmaz hızlı çalışan bir mimaridir.
Özellikle sosyal medya ve uygulama yorumları gibi yazım hataları içeren (sub-word bilgisine ihtiyaç duyan) verilerde çok başarılıdır.

## Bölüm 1 — Ortam Kurulumu ve Veri Hazırlama

In [ ]:
import pandas as pd
import numpy as np
import fasttext
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# FastText özel bir veri formatı ister: __label__SINIF metin
df = pd.read_csv('data/processed/reviews_cleaned.csv').dropna(subset=['cleaned_text', 'label'])
min_class_size = df['label'].value_counts().min()
df_balanced = df.groupby('label').apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)

# FastText formatına dönüştürme
df_balanced['fasttext_format'] = '__label__' + df_balanced['label'] + ' ' + df_balanced['cleaned_text'].astype(str)

train_data, test_data = train_test_split(df_balanced['fasttext_format'], test_size=0.2, random_state=42, stratify=df_balanced['label'])

# Verileri txt dosyası olarak diske kaydetme (FastText dosyadan okur)
with open('data/processed/fasttext_train.txt', 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(f"{item}\n")

with open('data/processed/fasttext_test.txt', 'w', encoding='utf-8') as f:
    for item in test_data:
        f.write(f"{item}\n")

print("FastText verileri hazırlandı.")


## Bölüm 2 — Model Eğitimi (Training)

In [ ]:
# Eğitimi başlat (Çalıştırmak için yorum satırlarını kaldırın)
print("FastText modeli eğitiliyor...")
model = fasttext.train_supervised(
    input='data/processed/fasttext_train.txt',
    lr=0.1,                # Öğrenme oranı
    epoch=25,              # Eğitim turu
    wordNgrams=2,          # Bi-gram (ikili kelime) özellikleri kullan
    dim=100                # Vektör boyutu
)
print("Eğitim tamamlandı!")


## Bölüm 3 — Değerlendirme ve Modeli Kaydetme

In [ ]:
# Test seti üzerinden başarıyı ölç
def test_fasttext(model, test_file):
    result = model.test(test_file)
    print(f"Örnek Sayısı: {result[0]}")
    print(f"Precision: {result[1]:.4f}")
    print(f"Recall: {result[2]:.4f}")

print("\n--- Test Sonuçları ---")
test_fasttext(model, 'data/processed/fasttext_test.txt')

# Modeli Kaydet
model.save_model("models/fasttext_model.bin")
print("\nModel 'models/fasttext_model.bin' olarak kaydedildi.")


## Bölüm 4 — Örnek Tahminler

In [ ]:
text = "harika bir uygulama herkese tavsiye ederim"
print(f"Metin: {text}")
print(f"Tahmin: {model.predict(text)}")
